# AlphaI × Polaris — BTC GBM Forecaster (IMPROVED v2)

**Part A**: 30-day walk-forward backtest → `backtest_results.jsonl`

## Key improvements over starter notebook
| What | Starter | v2 |
|------|---------|----|
| Vol estimator | Rolling std (20 bars) | **EWMA λ=0.94 (RiskMetrics)** |
| nu estimation | Single MLE on whole history | **Sliding MLE on 120-bar window** |
| Drift | Zero | **Itô-corrected: μ - ½σ²** |
| Simulations | 10,000 | **15,000** (smoother quantiles) |
| Warmup | 120 bars | **150 bars** (more stable nu) |
| Regime tracking | No | **Yes (calm/neutral/volatile)** |

Run all cells top-to-bottom. Cell 6 prints metrics and downloads the JSONL.

In [ ]:
# ── Cell 1: Install / imports ──────────────────────────────────────────────
!pip install -q scipy

import requests, json, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Cell 2: Fetch BTCUSDT 1-hour data ─────────────────────────────────────
BINANCE_BASE = 'https://data-api.binance.vision/api/v3/klines'

def fetch_btc_hourly(n_bars=900):
    """
    Fetch the last n_bars of BTCUSDT 1h candles.
    Uses data-api.binance.vision — no geo-block, no API key.
    We fetch 900 so we have 150-bar warmup + 750 prediction bars.
    """
    all_bars, end_time = [], None
    while len(all_bars) < n_bars:
        params = {'symbol':'BTCUSDT','interval':'1h',
                  'limit': min(1000, n_bars - len(all_bars))}
        if end_time:
            params['endTime'] = end_time
        resp = requests.get(BINANCE_BASE, params=params, timeout=15)
        resp.raise_for_status()
        bars = resp.json()
        if not bars: break
        all_bars = bars + all_bars
        end_time = bars[0][0] - 1
        if len(bars) < 1000: break
        time.sleep(0.2)
    df = pd.DataFrame(all_bars, columns=[
        'open_time','open','high','low','close','volume',
        'close_time','quote_vol','n_trades','taker_buy_base','taker_buy_quote','ignore'])
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    for c in ['open','high','low','close','volume']:
        df[c] = df[c].astype(float)
    df = df.sort_values('open_time').reset_index(drop=True)
    return df[['open_time','open','high','low','close','volume']]

df = fetch_btc_hourly(900)
print(f'Fetched {len(df)} bars')
print(f'Range: {df.open_time.iloc[0]}  →  {df.open_time.iloc[-1]}')
df.tail(3)

In [ ]:
# ── Cell 3: IMPROVED Model — EWMA vol + adaptive nu ───────────────────────
#
# Three key ideas (same as before, implemented better):
# 1. NO PEEKING  — function only receives history up to bar i
# 2. VOLATILITY CLUSTERING — EWMA λ=0.94 (RiskMetrics), not rolling std
# 3. FAT TAILS — Student-t with MLE nu fit on sliding window
#
# Additional improvements:
# • Itô drift correction: GBM uses mu - 0.5*sigma² as drift
# • nu clipped to [2.5, 6] — BTC empirical range, prevents numerical issues
# • 15k simulations vs 10k — smoother quantile estimates

EWMA_LAMBDA = 0.94   # RiskMetrics industry standard
VOL_LOOKBACK = 24    # EWMA window length (effective half-life ~12 bars)
LOOKBACK     = 120   # window for nu MLE fit
N_SIM        = 15_000
WARMUP       = 150   # minimum bars before first prediction


def ewma_vol(log_returns, lam=EWMA_LAMBDA):
    """
    EWMA (Exponentially Weighted Moving Average) volatility.
    σ²_t = λ·σ²_{t-1} + (1-λ)·r²_{t-1}
    
    WHY better than rolling std:
    - Gives more weight to recent returns (recency bias is actually GOOD for vol)
    - Reacts faster to volatility spikes — widens range within 1-2 bars
    - No cliff-edge: old data decays smoothly rather than dropping off sharply
    - λ=0.94 is RiskMetrics (JP Morgan) standard for daily/hourly data
    """
    if len(log_returns) < 2:
        return float(np.std(log_returns, ddof=1)) if len(log_returns) else 1e-4
    var = float(np.var(log_returns[:min(10, len(log_returns))], ddof=1))  # seed
    for r in log_returns:
        var = lam * var + (1 - lam) * r**2
    return float(np.sqrt(max(var, 1e-12)))


def fit_nu_mle(log_returns):
    """
    MLE for Student-t degrees of freedom on a sliding window.
    Clipped to [2.5, 6]: BTC empirically sits here.
    
    WHY clip:
    - nu < 2.5 → infinite variance (numerically unstable)
    - nu > 6   → near-Gaussian (loses fat-tail protection)
    - BTC returns show nu ≈ 3-5 historically
    """
    if len(log_returns) < 20:
        return 4.0
    scale = np.std(log_returns, ddof=1)
    if scale < 1e-10:
        return 4.0
    try:
        nu, _, _ = student_t.fit(log_returns, floc=0, fscale=scale)
        return float(np.clip(nu, 2.5, 6.0))
    except Exception:
        return 4.0


def detect_regime(log_returns, short_w=12, long_w=72):
    """Simple regime detector: volatile if short-term vol > long-term vol."""
    if len(log_returns) < long_w:
        return 'neutral'
    v_short = np.std(log_returns[-short_w:], ddof=1)
    v_long  = np.std(log_returns[-long_w:],  ddof=1)
    ratio = v_short / (v_long + 1e-10)
    if ratio > 1.25: return 'volatile'
    if ratio < 0.80: return 'calm'
    return 'neutral'


def fit_and_predict(closes, n_sim=N_SIM, conf=0.95):
    """
    Given 1-D array of closing prices (most-recent last),
    return (low, high, current_price, vol_ann, nu, regime).
    
    STRICT NO-PEEK: caller must only pass history up to bar i.
    """
    log_ret = np.diff(np.log(closes))

    # ── EWMA volatility (hourly) ──────────────────────────────────────────
    vol_h = ewma_vol(log_ret[-max(VOL_LOOKBACK * 2, 50):])

    # ── Adaptive nu via MLE on sliding window ─────────────────────────────
    fit_data = log_ret[-min(LOOKBACK, len(log_ret)):]
    nu = fit_nu_mle(fit_data)

    # ── Itô drift correction ──────────────────────────────────────────────
    # In GBM, S_t = S_0 * exp((mu - sigma²/2)*t + sigma*W_t)
    # The -sigma²/2 term is the Itô correction.
    # Using raw mean log returns as drift is technically correct already,
    # but we subtract half-variance to avoid upward bias in price simulations.
    mu = float(np.mean(fit_data)) - 0.5 * vol_h**2

    # ── Regime ────────────────────────────────────────────────────────────
    regime = detect_regime(log_ret)

    # ── GBM simulation ────────────────────────────────────────────────────
    S0 = closes[-1]
    sim_ret    = student_t.rvs(df=nu, loc=mu, scale=vol_h, size=n_sim)
    sim_prices = S0 * np.exp(sim_ret)

    alpha = (1 - conf) / 2
    low   = float(np.quantile(sim_prices, alpha))
    high  = float(np.quantile(sim_prices, 1 - alpha))
    vol_ann = vol_h * np.sqrt(8760)  # hourly → annualized

    return low, high, float(S0), vol_ann, nu, regime


# Quick sanity check on last bar
l, h, s, vol_ann, nu, regime = fit_and_predict(df['close'].values)
print(f'Current price : ${s:,.2f}')
print(f'95% range     : ${l:,.2f}  –  ${h:,.2f}')
print(f'Width         : ${h-l:,.0f}  ({(h-l)/s*100:.2f}%)')
print(f'Vol (annual.) : {vol_ann*100:.1f}%')
print(f'Student-t ν   : {nu:.2f}  (BTC typical: 3–5)')
print(f'Regime        : {regime}')

In [ ]:
# ── Cell 4: Winkler score + evaluate() ────────────────────────────────────
# (Unchanged — these are provided by the challenge spec)

def winkler_score(low, high, actual, alpha=0.05):
    """
    Winkler interval score (lower = better forecaster).
      inside range  → score = width
      below range   → score = width + (2/alpha)*(low - actual)
      above range   → score = width + (2/alpha)*(actual - high)
    """
    width = high - low
    if   actual < low:  return width + (2/alpha)*(low - actual)
    elif actual > high: return width + (2/alpha)*(actual - high)
    return width

def evaluate(predictions):
    """
    predictions: list of dicts with keys 'low', 'high', 'actual'.
    Returns dict with coverage_95, mean_width, mean_winkler_95.
    """
    hits, widths, winklers = [], [], []
    for p in predictions:
        low, high, actual = p['low'], p['high'], p['actual']
        hits.append(1 if low <= actual <= high else 0)
        widths.append(high - low)
        winklers.append(winkler_score(low, high, actual))
    return {
        'coverage_95':     float(np.mean(hits)),
        'mean_width':      float(np.mean(widths)),
        'mean_winkler_95': float(np.mean(winklers)),
        'n_predictions':   len(predictions),
    }

print('evaluate() ready')

In [ ]:
# ── Cell 5: Part A — 30-day walk-forward backtest (IMPROVED) ──────────────
#
# Same strict no-peek structure as starter, but:
# • Warmup = 150 bars (more stable EWMA + nu)
# • Records vol, nu, regime per bar for analysis
#
# For each bar i (starting at index `WARMUP`):
#   • history = closes[0 .. i]     ← NO bar i+1 (strict no-peeking)
#   • actual  = closes[i+1]        ← the future we predict

closes     = df['close'].values
timestamps = df['open_time'].tolist()

predictions = []
total = len(closes) - WARMUP - 1
print(f'Running {total} predictions (WARMUP={WARMUP}) …')

for i in range(WARMUP, len(closes) - 1):
    # ── STRICT NO-PEEK: only use indices 0..i ──────────────────────────
    history = closes[:i+1]     # does NOT include closes[i+1]
    actual  = float(closes[i+1])

    try:
        low, high, s0, vol_ann, nu, regime = fit_and_predict(history)
    except Exception as e:
        print(f'  [WARN] bar {i}: {e}')
        continue

    w = winkler_score(low, high, actual)
    predictions.append({
        'bar_index':       int(i),
        'prediction_time': timestamps[i].isoformat(),
        'target_time':     timestamps[i+1].isoformat(),
        'current_price':   float(s0),
        'low':             low,
        'high':            high,
        'actual':          actual,
        'hit':             bool(low <= actual <= high),
        'width':           high - low,
        'winkler':         w,
        'vol_ann':         vol_ann,
        'nu':              nu,
        'regime':          regime,
    })

    done = i - WARMUP
    if done % 100 == 0:
        cov = np.mean([p['hit'] for p in predictions])
        print(f'  {done}/{total} … running coverage={cov:.4f}')

print(f'Done. {len(predictions)} predictions collected.')

In [ ]:
# ── Cell 6: Metrics + per-regime breakdown + save JSONL ───────────────────
metrics = evaluate(predictions)

print('══ Backtest Metrics (IMPROVED v2) ═══════════════════════════════')
print(f"  Coverage @95%  : {metrics['coverage_95']:.4f}  (ideal ≈ 0.95)")
print(f"  Mean width     : ${metrics['mean_width']:>12,.2f}")
print(f"  Mean Winkler ↓ : ${metrics['mean_winkler_95']:>12,.2f}  (lower = better)")
print(f"  N predictions  : {metrics['n_predictions']}")
print('════════════════════════════════════════════════════════════════')

# Per-regime breakdown
print('\nPer-regime breakdown:')
for regime in ['calm', 'neutral', 'volatile']:
    subset = [p for p in predictions if p.get('regime') == regime]
    if subset:
        cov = np.mean([p['hit'] for p in subset])
        wkl = np.mean([p['winkler'] for p in subset])
        wid = np.mean([p['width'] for p in subset])
        print(f"  {regime:8s}: n={len(subset):3d}, cov={cov:.4f}, "
              f"width=${wid:,.0f}, winkler=${wkl:,.0f}")

# Save JSONL
out_file = 'backtest_results.jsonl'
with open(out_file, 'w') as f:
    for row in predictions:
        f.write(json.dumps(row) + '\n')

print(f'\nSaved to {out_file}')

# In Colab, trigger download
try:
    from google.colab import files
    files.download(out_file)
except ImportError:
    print('(Not in Colab — file saved to current directory)')

In [ ]:
# ── Cell 7: Rich visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df_pred = pd.DataFrame(predictions)
df_pred['prediction_time'] = pd.to_datetime(df_pred['prediction_time'])
df_pred['hit'] = df_pred['hit'].astype(bool)

BG = '#0e1117'
CARD = '#1a1d24'
ORANGE = '#f7931a'
GREEN = '#26a69a'
RED = '#ef5350'
BLUE = '#63b3ed'
PURPLE = '#b39ddb'

fig = plt.figure(figsize=(16, 12), facecolor=BG)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

def style_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors='#ccc')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
    ax.title.set_color('#fafafa')
    ax.xaxis.label.set_color('#aaa')
    ax.yaxis.label.set_color('#aaa')

# ── Plot 1: Price + range ribbon (last 200 bars) ──────────────────────────
ax1 = fig.add_subplot(gs[0, :])
style_ax(ax1)
last = df_pred.tail(200)
ax1.fill_between(last['prediction_time'], last['low'], last['high'],
                 alpha=0.2, color=ORANGE, label='95% range')
ax1.plot(last['prediction_time'], last['actual'],
         color='#f6c90e', lw=1.3, label='Actual BTC')
# Shade regimes
for regime, color in [('volatile', RED), ('calm', GREEN)]:
    mask = last['regime'] == regime
    if mask.any():
        ax1.fill_between(last['prediction_time'], last['actual'].min(), last['actual'].max(),
                         where=mask, alpha=0.07, color=color, label=f'Regime: {regime}')
ax1.set_title('BTC actual vs 95% predicted range (last 200 bars) with regime shading')
ax1.legend(facecolor=CARD, labelcolor='#ccc', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Plot 2: Hits/misses ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
style_ax(ax2)
hits_df   = df_pred[df_pred['hit']]
misses_df = df_pred[~df_pred['hit']]
ax2.scatter(hits_df['prediction_time'],   hits_df['actual'],
            color=GREEN, s=8, alpha=0.5, label='Hit')
ax2.scatter(misses_df['prediction_time'], misses_df['actual'],
            color=RED, s=18, marker='x', label='Miss')
ax2.set_title(f'Hits (green) vs misses (red) — Coverage: {metrics["coverage_95"]:.4f}')
ax2.legend(facecolor=CARD, labelcolor='#ccc', fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Plot 3: Width over time ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
style_ax(ax3)
ax3.plot(df_pred['prediction_time'], df_pred['width'], color=PURPLE, lw=0.8)
ax3.set_title('Range width (EWMA vol clustering visible)')
ax3.set_ylabel('Width ($)', color='#ccc')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Plot 4: nu over time ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
style_ax(ax4)
ax4.plot(df_pred['prediction_time'], df_pred['nu'], color=BLUE, lw=0.9)
ax4.axhline(4.0, color='#555', linestyle='--', lw=0.8, label='ν=4 reference')
ax4.set_title('Student-t ν (degrees of freedom) over time')
ax4.set_ylabel('ν', color='#ccc')
ax4.set_ylim(2, 6.5)
ax4.legend(facecolor=CARD, labelcolor='#ccc', fontsize=8)

# ── Plot 5: Regime pie ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor(CARD)
regime_counts = df_pred['regime'].value_counts()
colors = {'calm': GREEN, 'neutral': BLUE, 'volatile': RED}
pie_colors = [colors.get(r, '#888') for r in regime_counts.index]
wedges, texts, autotexts = ax5.pie(
    regime_counts.values,
    labels=regime_counts.index,
    colors=pie_colors,
    autopct='%1.1f%%',
    textprops={'color': '#ccc'},
)
for at in autotexts:
    at.set_color('#111')
ax5.set_title('Volatility regime distribution', color='#fafafa')

plt.savefig('backtest_chart_v2.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print('Chart saved as backtest_chart_v2.png')

# Trigger Colab download
try:
    from google.colab import files
    files.download('backtest_chart_v2.png')
except ImportError:
    pass

## Submit
1. **Copy your metrics** from Cell 6 into the submission form.
2. **Download** `backtest_results.jsonl` and commit it to your GitHub repo.
3. **Deploy** `dashboard.py` (v2) to Streamlit Community Cloud.
4. **Fill** the [submission form](https://docs.google.com/forms/d/e/1FAIpQLSe93W4OX2z08uA_pkGddMRSJk9mb4F7d2mXozNez8kT4hyXwQ/viewform).

## What to highlight in your submission
- EWMA vol (λ=0.94) replaces rolling std → better vol clustering response
- Adaptive nu via MLE per sliding window → more accurate tail modeling
- Itô drift correction applied
- Regime tracking (calm/volatile/neutral) for analytical insight
- Part C persistence with deduplication guard